# 01 — Colab 환경 검증

**작전**: TF 1.x는 Python 3.12에 설치 불가 → Colab 기본 TF 2.x + `tf.compat.v1` 모드로 SC-FEGAN 동작.
MediaPipe는 최신 0.10.35 솔루션 모듈 빠짐 → **0.10.21 핀** 사용.

**전제**:
- 런타임 → 런타임 유형 변경 → 하드웨어 가속기: **T4 GPU**
- Drive에 `MyDrive/cv-final/` 폴더 업로드 완료

**체크리스트**:
- [ ] GPU 인식 (`nvidia-smi`)
- [ ] TF 2.x + compat 모드 동작
- [ ] MediaPipe 0.10.21 + Face Mesh 478점 검출
- [ ] 우리 project/ 코드 import 동작

## 1. GPU 확인

In [ ]:
!nvidia-smi

## 2. TF 환경 (compat 모드)

TF 1.x 설치 시도는 건너뜀 — Python 3.12 호환 wheel 없음.
Colab 기본 TF 2.x를 `tf.compat.v1`로 운용.

In [ ]:
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)
print('Devices:', tf.config.list_physical_devices())

In [ ]:
# matmul GPU 테스트 — TF 2.x compat 모드
import tensorflow as tf
tf_v1 = tf.compat.v1 if tf.__version__.startswith('2.') else tf
tf_v1.disable_eager_execution()
print('TF1 compat mode OK, TF version:', tf.__version__)

a = tf_v1.constant([[1.0, 2.0], [3.0, 4.0]])
b = tf_v1.constant([[5.0, 6.0], [7.0, 8.0]])
c = tf_v1.matmul(a, b)

with tf_v1.Session() as sess:
    print('matmul result:', sess.run(c))

## 3. MediaPipe + OpenCV

⚠️ **MediaPipe Tasks API 사용**. legacy `mp.solutions.face_mesh`는 protobuf 충돌로 깨짐 → Tasks API 로 대체. 478점 추출은 동일.

In [ ]:
!pip install --quiet mediapipe==0.10.21 opencv-python pyyaml

In [ ]:
import cv2, mediapipe as mp, yaml, numpy as np
print('cv2:', cv2.__version__)
print('mediapipe:', mp.__version__)
print('numpy:', np.__version__)
print('pyyaml:', yaml.__version__)

# Tasks API 사용 가능 여부
from mediapipe.tasks.python import vision as mp_vision
print('Tasks API OK:', mp_vision.FaceLandmarker is not None)

In [ ]:
# Face Landmarker smoke test (Tasks API)
import os, urllib.request

# SC-FEGAN 샘플 이미지
!test -f SC-FEGAN/samples/00001.png || git clone https://github.com/run-youngjoo/SC-FEGAN.git

# face_landmarker.task 모델 다운로드 (~3.7MB)
MODEL_PATH = '/tmp/face_landmarker.task'
if not os.path.exists(MODEL_PATH):
    url = ('https://storage.googleapis.com/mediapipe-models/face_landmarker/'
           'face_landmarker/float16/1/face_landmarker.task')
    print('Downloading face_landmarker.task ...')
    urllib.request.urlretrieve(url, MODEL_PATH)
    print('✅ 다운로드 완료')
else:
    print('✅ 모델 캐시됨')

# Tasks API로 검출
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
options = mp_vision.FaceLandmarkerOptions(
    base_options=base_options,
    num_faces=1,
    running_mode=mp_vision.RunningMode.IMAGE,
)
detector = mp_vision.FaceLandmarker.create_from_options(options)

img_bgr = cv2.imread('SC-FEGAN/samples/00001.png')
print('image shape:', img_bgr.shape if img_bgr is not None else None)

rgb = np.ascontiguousarray(img_bgr[..., ::-1])
mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
result = detector.detect(mp_image)

print('faces detected:', len(result.face_landmarks))
if result.face_landmarks:
    print('landmarks count:', len(result.face_landmarks[0]))

## 4. 우리 project/ 코드 import 검증

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 본인 드라이브 경로에 맞춰 수정
PROJECT_ROOT = '/content/drive/MyDrive/cv-final'
import sys
sys.path.insert(0, PROJECT_ROOT)

In [ ]:
from project.face_analyzer import detect_landmarks, normalize_face, compute_ratios
from project.recommender import recommend, load_procedures
from project.input_generator import make_mask

img = cv2.imread('SC-FEGAN/samples/00001.png')
aligned, meta = normalize_face(img)
print('aligned shape:', aligned.shape)

lm = detect_landmarks(aligned)
print('landmarks:', lm.shape)

ratios = compute_ratios(lm)
for k, v in ratios.items():
    print(f'  {k}: {v:.3f}')

In [ ]:
procs = load_procedures(f'{PROJECT_ROOT}/project/recommender/procedures.yaml')
recs = recommend(ratios, procs)
for r in recs:
    print(f'[{r.confidence:.2f}] {r.name_ko}: ₩{r.price_krw_min:,} ~ ₩{r.price_krw_max:,}')
    print(f'    {r.reasoning}')

if recs:
    mask = make_mask(lm, recs[0].procedure)
    print('mask shape:', mask.shape, 'non-zero:', int((mask > 0).sum()))

## 5. 결과 시각화

In [ ]:
import matplotlib.pyplot as plt

vis = aligned.copy()
for (x, y) in lm:
    cv2.circle(vis, (int(x), int(y)), 1, (0, 255, 0), -1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(aligned[..., ::-1]); axes[0].set_title('Normalized')
axes[1].imshow(vis[..., ::-1]); axes[1].set_title('Landmarks 478')
if recs:
    axes[2].imshow(mask[..., 0], cmap='gray'); axes[2].set_title(f'Mask: {recs[0].name_ko}')
for ax in axes: ax.axis('off')
plt.show()

## 완료 기준

- ✅ `nvidia-smi` GPU 인식
- ✅ TF version 2.x + matmul 결과 `[[19. 22.] [43. 50.]]`
- ✅ MediaPipe `has solutions: True`
- ✅ Face Mesh `faces detected: 1, landmarks count: 478`
- ✅ 비율 10개 + 추천 1건 이상 + mask non-zero > 0
- ✅ 시각화 이미지 3장 표시

→ 새세션을 새로 시작하고 `02_scfegan_smoke.ipynb` 진행